# Cascade. Protocol-matched comparison on DDR

DDR is the only dataset in the test pool with an official test split that is widely used for
grading comparisons: 4,105 images, 346 of them ungradable. Published results use two
protocols:

- *five-class*: Grades 0-4 on the 3,759 gradable images;
- *six-class*: all 4,105 images, ungradable as a class.

Our outputs are end-to-end (gate first; 154 DDR disease cases never reach the grader). A
five-class protocol has no slot for an *ungradable* prediction on a gradable image, so those
predictions are scored three ways: mapped to Grade 4 (our definition), dropped, or mapped to
the grade farthest from the reference (worst case).

**Steps.**
1. Recompute the cascade numbers (same bootstrap stream as notebook 01, so the CIs match).
2. Five-class and six-class rows against published results.
3. Six-class per-class recall.

## 0. Configuration

In [ ]:
from pathlib import Path

# Inputs (all per-image CSVs):
#   data/test/test_grade.csv                image, grade (0..5) for all 58,689 test images
#   data/test/stage1_manifest.csv           stem, prob, pred_binary (Stage I routing; 1 = passed)
#   outputs/stage3/eval/oof_corn.csv        OOF predictions (stage3_grading/02)
#   outputs/stage3/eval/lgdrg_test_predictions.csv   ensemble predictions (stage3_grading/03)
DATA_ROOT = Path("data")
GT_CSV = DATA_ROOT / "test" / "test_grade.csv"
MANIFEST_CSV = DATA_ROOT / "test" / "stage1_manifest.csv"

OUT_ROOT = Path("outputs")
OOF_CSV = OUT_ROOT / "stage3" / "eval" / "oof_corn.csv"
PRED_CSV = OUT_ROOT / "stage3" / "eval" / "lgdrg_test_predictions.csv"
CASCADE_DIR = OUT_ROOT / "cascade"
CASCADE_DIR.mkdir(parents=True, exist_ok=True)

GATE_THRESHOLD = 0.2391715943813324     # frozen Stage I operating point

import pandas as pd
from hierarchiretina.stage3.cascade import run_final_evaluation
from hierarchiretina.stage3.metrics import five_class_metrics, six_class_recall

## 1. Cascade numbers

In [ ]:
res = run_final_evaluation(OOF_CSV, PRED_CSV, GT_CSV, MANIFEST_CSV, CASCADE_DIR,
                           gate_threshold=GATE_THRESHOLD)
d = res["ddr"]
C = res["tables"]["cascade"]
D = C[C.source == "ddr"]
print({k: v for k, v in d.items() if not isinstance(v, tuple)})

## 2. Matched-protocol table (published values as reported by their authors)

In [ ]:
rows = [
    ["FA+KC-Net+R2 (Tian et al., 2023)", "5-class, grader", 84.49, 0.862],
    ["RadFuse (Mohsen et al., 2025)", "5-class, grader", 83.32, 0.855],
    ["Dual-SwinOrd (Yu et al., 2026)", "official split, grader", 86.54, 0.904],
    ["Ours, ungradable -> G4", "5-class, end-to-end", 100 * d["acc5_pred5to4"], d["qwk5_pred5to4"]],
    [f"Ours, ungradable dropped (n={d['n_drop_pred5']:,})", "5-class, end-to-end", None,
     d["qwk5_drop_pred5"]],
    ["Ours, ungradable -> worst case", "5-class, end-to-end", 100 * d["acc5_worstcase"],
     d["qwk5_worstcase"]],
    ["FA+KC-Net+R2 (Tian et al., 2023)", "6-class, grader", 83.78, None],
    ["Ours", "6-class, end-to-end", 100 * d["acc6_all"], None],
]
print(f"five-class QWK 95% CI: {d['qwk5_pred5to4_ci'][0]:.3f}-{d['qwk5_pred5to4_ci'][1]:.3f}; "
      f"ungradable predictions on gradable DDR images: {d['pred5_on_gradable']}")
pd.DataFrame(rows, columns=["method", "protocol", "acc_%", "QWK"]).round(3)

Five-class accuracy with a predicted Grade 5 counted as an error (`acc_strict`) or mapped
to 4 (`acc_clipped`, reported above), for DDR and the other sources.

In [ ]:
pd.DataFrame({s.upper(): five_class_metrics(C[C.source == s].true_grade, C[C.source == s].final)
              for s in ["ddr", "eyepacs", "aptos", "fgadr", "messidor", "idrid"]}).T.round(4)

## 3. Six-class per-class recall on DDR (published rows: Tian et al., 2023, Table 9)

In [ ]:
PUBLISHED = [
    ("VGG16", [95.37, 4.23, 56.23, 39.40, 64.36, 94.22], 75.20),
    ("ResNet-18", [95.48, 5.82, 62.20, 36.62, 58.18, 91.33], 76.59),
    ("SE-BN-Inception", [94.52, 4.76, 64.58, 12.68, 58.18, 90.46], 76.39),
    ("ResNet50+CBAM", [99.10, 1.59, 64.21, 40.85, 61.46, 95.36], 79.34),
    ("FA-Net", [96.92, 22.22, 70.54, 30.99, 67.27, 97.40], 81.75),
    ("FA+KC-Net+R1", [96.22, 28.04, 74.48, 35.21, 67.27, 97.40], 83.07),
    ("FA+KC-Net+R2", [94.79, 52.91, 74.93, 39.44, 67.27, 97.40], 83.78),
]
r6 = six_class_recall(D.true_grade, D.final)
assert r6["n"] == 4105, "the DDR slice is not the official test split"
cols = ["DR 0", "DR 1", "DR 2", "DR 3", "DR 4", "Ungradable"]
tab = pd.DataFrame([dict(model=n, **dict(zip(cols, pc)), OA=oa) for n, pc, oa in PUBLISHED]
                   + [dict(model="LG-DRG cascade (ours)", **dict(zip(cols, 100 * r6["per_class_recall"])),
                           OA=100 * r6["oa"])])
tab.round(2)